In [1]:
!pip install pandas


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
analyst_ratings = pd.read_csv('../data/raw_analyst_ratings.csv')

In [ ]:
print(analyst_ratings.head())
print(analyst_ratings.info())
print(analyst_ratings.describe())

In [ ]:
analyst_ratings['headline_length'] = analyst_ratings['headline'].apply(len)
articles_per_publisher = analyst_ratings['publisher'].value_counts()
publication_trends = analyst_ratings['date'].value_counts()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# Get feature names
feature_names = vectorizer.get_feature_names_out()

# Convert sparse matrix to DataFrame selectively, if needed
# For example, to get top N features
top_n = 20
sum_words = X.sum(axis=0)
words_freq = [(word, sum_words[0, idx]) for word, idx in zip(feature_names, range(len(feature_names)))]
words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)[:top_n]
print(words_freq)

In [ ]:
print(analyst_ratings['date'].unique())

In [ ]:
# Headline Length Stats
analyst_ratings['headline_length'] = analyst_ratings['headline'].astype(str).str.len()
print(analyst_ratings['headline_length'].describe())

In [ ]:
# Trend Over Time
analyst_ratings['date'] = pd.to_datetime(analyst_ratings['date'], errors='coerce')
articles_per_day = analyst_ratings['date'].dt.date.value_counts().sort_index()
articles_per_day.plot(title='Articles Published Over Time', figsize=(12, 5))

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from nltk.corpus import stopwords
import nltk

# One-time download
nltk.download('stopwords')

# Step 1: Sample the data (5000 headlines or fewer)
sampled_texts = analyst_ratings['headline'].dropna().sample(n=5000, random_state=42)

# Step 2: Vectorize with limited vocabulary
vectorizer = CountVectorizer(
    stop_words=stopwords.words('english'),
    max_df=0.95,
    min_df=2,
    max_features=3000  # reduce vocabulary size
)
X = vectorizer.fit_transform(sampled_texts)

# Step 3: Fit LDA model (fewer topics, multi-core)
lda = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    learning_method='batch',  # or 'online' for larger batches
    n_jobs=-1  # use all available CPU cores
)
lda.fit(X)

# Step 4: Display top words in each topic
for idx, topic in enumerate(lda.components_):
    top_words = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-10:]]
    print(f"Topic {idx+1}: {', '.join(top_words)}")

In [ ]:
# Publication frequency over time
analyst_ratings.set_index('date', inplace=True)
analyst_ratings['headline'].resample('D').count().plot(title='Publication Frequency Over Time', figsize=(12, 5))

# Time of Day analysis
analyst_ratings['hour'] = analyst_ratings.index.hour
analyst_ratings['hour'].value_counts().sort_index().plot(kind='bar', title='Article Frequency by Hour', figsize=(10, 4))

In [ ]:
# Top Publishers
print(analyst_ratings['publisher'].value_counts().head(10))

In [ ]:
# Most common words per top publisher
top_publishers = analyst_ratings['publisher'].value_counts().head(5).index
for pub in top_publishers:
    pub_text = analyst_ratings[analyst_ratings['publisher'] == pub]['headline'].dropna().str.cat(sep=' ')
    words = pd.Series(pub_text.lower().split())
    print(f"\n{pub}:")
    print(words.value_counts().head(10))

In [ ]:
# Email domain extraction (if publisher is an email)
analyst_ratings['domain'] = analyst_ratings['publisher'].str.extract(r'@([\w\.-]+)')
print(analyst_ratings['domain'].value_counts().head(10))

In [ ]:
op_publishers = analyst_ratings['publisher'].value_counts().head(10)
top_publishers.plot(kind='barh', figsize=(10, 5), title='Top 10 Publishers by Article Count')